# Thick-Walled Cylinder — pyGmshViewer Integration (v2)

This notebook extends the `example_plate_pyGmsh` workflow by adding:

1. **VTK Export** — write mesh + results to `.vtu` via the `VTKExport` class
2. **pyGmshViewer** launch — standalone Qt/VTK viewer with contours, deformed shapes, and probes
3. **Programmatic probing** — sample field values at points, along lines, and on cut planes

## v2 Changes

Updated to use the current pyGmsh API:

- **`renumber_mesh()` + `get_fem_data()`** — single FEMData object replaces the old `get_numbered_mesh()`
- **FEMData convenience methods** — `.nodes()`, `.elements()`, `.node_index()`,
  `.get_node_coords()` replace manual maps and index lookups
- **`fem.physical`** — boundary/load queries live on the FEMData snapshot, no separate `g.physical` calls
- **`VTKExport(g)` class** — streamlined VTK export while Gmsh is still alive

In [ ]:
from pyGmsh import pyGmsh
from pyGmsh.VTKExport import VTKExport, write_vtu, VTK_TRIANGLE
import numpy as np
import openseespy.opensees as ops

## 1 — Geometry & Mesh

Same Lamé problem as `example_plate_pyGmsh.ipynb`. After meshing, we call
`renumber_mesh(method='rcm')` to get contiguous solver-ready IDs, then
`get_fem_data()` to snapshot everything into one `FEMData` object.

In [ ]:
# ============================================================
# Parameters
# ============================================================
inner_radius = 100.0    # [mm]
outer_radius = 200.0    # [mm]
lc           = 10.0     # [mm] mesh size
E   = 210.0e3           # [MPa]
nu  = 0.3
p   = 100.0             # [MPa] internal pressure
thk = 1.0               # [mm]

# ============================================================
# pyGmsh: Geometry + Mesh
# ============================================================
g = pyGmsh(model_name="Plate2D", verbose=False)
g.initialize()

pc = g.model.add_point(0, 0, 0, lc=lc)
p1 = g.model.add_point(inner_radius, 0, 0, lc=lc)
p2 = g.model.add_point(outer_radius, 0, 0, lc=lc)
p3 = g.model.add_point(0, outer_radius, 0, lc=lc)
p4 = g.model.add_point(0, inner_radius, 0, lc=lc)

l1 = g.model.add_line(p1, p2)
l2 = g.model.add_arc(p2, pc, p3)
l3 = g.model.add_line(p3, p4)
l4 = g.model.add_arc(p4, pc, p1)

loop = g.model.add_curve_loop([l1, l2, l3, l4])
surf = g.model.add_plane_surface(loop)

pg_symY = g.physical.add(1, [l1], name="Sym_Y")
pg_symX = g.physical.add(1, [l3], name="Sym_X")
pg_pres = g.physical.add(1, [l4], name="Pressure")
pg_plat = g.physical.add(2, [surf], name="Plate")

g.mesh.set_order(1)
g.mesh.generate(2)

# Renumber to contiguous solver-ready IDs (RCM for bandwidth reduction)
g.mesh.renumber_mesh(method='rcm')

# Extract the single FEMData object — this is our sole data source
fem = g.mesh.get_fem_data(dim=2)
print(fem.info)
fem.physical.summary()

In [ ]:
# ============================================================
# OpenSees: Build model, apply loads, solve
# ============================================================
# After renumber_mesh(), fem.node_ids and fem.element_ids ARE
# the solver IDs — no Gmsh↔solver translation needed.
# ============================================================

ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 2)

# --- Nodes (using FEMData iterator) ---
for nid, coords in fem.nodes():
    ops.node(nid, *coords)

# --- Material ---
ops.nDMaterial("ElasticIsotropic", 1, E, nu)

# --- Elements (using FEMData iterator) ---
for eid, conn in fem.elements():
    ops.element("tri31", eid, *conn, thk, "PlaneStrain", 1)

# --- Boundary conditions (from fem.physical) ---
for nid in fem.physical.get_nodes(1, pg_symY)['tags']:
    ops.fix(nid, 0, 1)   # Sym_Y: fix uy, free ux

for nid in fem.physical.get_nodes(1, pg_symX)['tags']:
    ops.fix(nid, 1, 0)   # Sym_X: fix ux, free uy

# --- Consistent pressure loads on inner arc ---
# Edge elements from fem.physical — already in solver IDs
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)

inner = fem.physical.get_elements(1, pg_pres)
inner_edges = inner['connectivity']   # (nEdge, 2)

nodal_forces = {}
for edge in inner_edges:
    n1, n2 = int(edge[0]), int(edge[1])

    # Coordinates via FEMData lookup
    x1, y1, _ = fem.get_node_coords(n1)
    x2, y2, _ = fem.get_node_coords(n2)

    Le = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    r1 = np.sqrt(x1**2 + y1**2)
    r2 = np.sqrt(x2**2 + y2**2)

    # Traction: t = p * (x/r, y/r)
    tx1, ty1 = p * x1/r1, p * y1/r1
    tx2, ty2 = p * x2/r2, p * y2/r2

    # Consistent nodal forces: F = integral(N^T * t dGamma)
    Fx1 = (Le/6) * (2*tx1 + tx2);  Fy1 = (Le/6) * (2*ty1 + ty2)
    Fx2 = (Le/6) * (tx1 + 2*tx2);  Fy2 = (Le/6) * (ty1 + 2*ty2)

    nodal_forces.setdefault(n1, [0., 0.])
    nodal_forces.setdefault(n2, [0., 0.])
    nodal_forces[n1][0] += Fx1;  nodal_forces[n1][1] += Fy1
    nodal_forces[n2][0] += Fx2;  nodal_forces[n2][1] += Fy2

for nid, (fx, fy) in nodal_forces.items():
    ops.load(nid, fx, fy)

# --- Solve ---
ops.constraints("Transformation")
ops.numberer("RCM")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1e-8, 10)
ops.algorithm("Newton")
ops.integrator("LoadControl", 1.0)
ops.analysis("Static")
ok = ops.analyze(1)
print("CONVERGED" if ok == 0 else "FAILED")

In [ ]:
# ============================================================
# Extract results into numpy arrays
# ============================================================
# After renumber_mesh(), fem IDs = solver IDs.
# No Gmsh↔solver translation — index directly via FEMData.
# ============================================================

nNode = fem.info.n_nodes
nElem = fem.info.n_elems

# -- Displacements --
disp = np.zeros((nNode, 2))
for i, nid in enumerate(fem.node_ids):
    disp[i, 0] = ops.nodeDisp(int(nid), 1)
    disp[i, 1] = ops.nodeDisp(int(nid), 2)

# -- Radial displacement --
r_nodes = np.sqrt(fem.node_coords[:, 0]**2 + fem.node_coords[:, 1]**2)
r_safe  = np.where(r_nodes > 1e-12, r_nodes, 1.0)
ur = (fem.node_coords[:, 0]*disp[:, 0] + fem.node_coords[:, 1]*disp[:, 1]) / r_safe

# -- Element stresses (CST: one constant stress per element) --
sig_xx = np.zeros(nElem)
sig_yy = np.zeros(nElem)
sig_xy = np.zeros(nElem)

for i, eid in enumerate(fem.element_ids):
    s = ops.eleResponse(int(eid), "stresses")
    sig_xx[i] = s[0]
    sig_yy[i] = s[1]
    sig_xy[i] = s[2]

# -- Nodal-averaged stress --
conn_idx = np.array([[fem.node_index(n) for n in row] for row in fem.connectivity])
sig_xx_nodal = np.zeros(nNode)
node_count   = np.zeros(nNode)

for e in range(nElem):
    for ln in range(3):
        nidx = conn_idx[e, ln]
        sig_xx_nodal[nidx] += sig_xx[e]
        node_count[nidx]   += 1.0

node_count[node_count == 0] = 1.0
sig_xx_nodal /= node_count

# -- Von Mises --
von_mises_elem = np.sqrt(sig_xx**2 - sig_xx*sig_yy + sig_yy**2 + 3*sig_xy**2)

print(f"u_r  range: [{ur.min():.6f}, {ur.max():.6f}] mm")
print(f"\u03c3_xx range: [{sig_xx.min():.2f}, {sig_xx.max():.2f}] MPa")
print(f"\u03c3_vm range: [{von_mises_elem.min():.2f}, {von_mises_elem.max():.2f}] MPa")

ops.wipe()
print("\nOpenSees wiped — results stored in numpy arrays.")

## 2 — Export to VTU for the Viewer

pyGmsh's `VTKExport` writes `.vtu` files (VTK UnstructuredGrid XML) that any
VTK-compatible tool can open: ParaView, VisIt, and our **pyGmshViewer**.

Two approaches:

| Approach | When to use |
|----------|-------------|
| `VTKExport(g)` class | While Gmsh is still running — extracts mesh internally, auto-detects cell type |
| `write_vtu()` function | Standalone — pass your own coords + connectivity arrays |

We use the **`VTKExport(g)` class** here — it handles 0-based connectivity
and VTK cell type detection automatically.

In [ ]:
from pathlib import Path

output_dir = Path("generated")
output_dir.mkdir(exist_ok=True)

# Build the VTK exporter from the live pyGmsh instance
vtk = VTKExport(g, dim=2)

# Add nodal fields
vtk.add_node_vector("Displacement", disp)          # (N, 2) → auto-pads to 3D
vtk.add_node_scalar("Displacement_Y", disp[:, 1])
vtk.add_node_scalar("Radial_Disp", ur)
vtk.add_node_scalar("Stress_XX_nodal", sig_xx_nodal)

# Add element fields
vtk.add_elem_scalar("Stress_XX", sig_xx)
vtk.add_elem_scalar("Stress_YY", sig_yy)
vtk.add_elem_scalar("Stress_XY", sig_xy)
vtk.add_elem_scalar("VonMises", von_mises_elem)

# Write
vtu_path = vtk.write(output_dir / "lame_plate_results.vtu")

print(vtk)
print(f"\nVTU written: {vtu_path}")

In [ ]:
# Finalize Gmsh (no longer needed — VTKExport already captured the mesh)
g.finalize()

## 3 — Launch pyGmshViewer

The viewer ships with a one-liner `show()` function that works from
notebooks and scripts alike:

```python
from pyGmshViewer import show

# Non-blocking (recommended in Jupyter — notebook keeps running)
show("results.vtu", blocking=False)

# Blocking (script-style — waits until you close the window)
show("results.vtu")

# Multiple files at once
show("mesh.vtu", "modes.pvd", blocking=False)
```

Or from the command line:
```bash
python -m pyGmshViewer generated/lame_plate_results.vtu
```

### Viewer Controls

| Feature | How |
|---------|-----|
| **Contour plot** | Click any field in the Model Tree |
| **Deformed shape** | Controls panel → check "Show Deformed Shape", set scale |
| **Point probe** | Ctrl+P or Probes panel → Point, then click on mesh |
| **Line probe** | Ctrl+L or Probes panel → Line, then two clicks |
| **Plane slice** | Probes panel → Slice X/Y/Z or Interactive Plane |
| **Camera** | XY, XZ, YZ, Iso toolbar buttons or drag to rotate |
| **Screenshot** | Ctrl+S |

In [ ]:
import sys
from pathlib import Path

# Add pyGmshViewer to path if not pip-installed
viewer_root = Path("..").resolve()
if str(viewer_root) not in sys.path:
    sys.path.insert(0, str(viewer_root))

from pyGmshViewer import show

# Launch the viewer (non-blocking so the notebook keeps running)
show(vtu_path, blocking=False)
print(f"Viewer launched with: {vtu_path}")

## 4 — Programmatic Probing (without the GUI)

You can use the probe engine directly in a script or notebook to sample
field values at specific locations. This is useful for:

- Extracting stress at a specific point (e.g., at the inner wall along x-axis)
- Plotting field variation along a line (e.g., radial stress profile)
- Comparing numerical vs analytical solutions at precise locations

The probes use VTK's interpolation, so they work at **any** location —
not just at mesh nodes.

In [ ]:
import pyvista as pv

# Load the VTU we just created
grid = pv.read(str(vtu_path))
print(f"Loaded: {grid.n_points} points, {grid.n_cells} cells")
print(f"Point fields: {list(grid.point_data.keys())}")
print(f"Cell fields:  {list(grid.cell_data.keys())}")

In [ ]:
# ============================================================
# Point Probe: sample at the inner wall along the x-axis
# ============================================================
# This is where the analytical solution gives maximum hoop stress.

probe_location = np.array([inner_radius, 0.0, 0.0])

# Create a single-point probe and sample the mesh
probe_pt = pv.PolyData(probe_location.reshape(1, 3))
sampled = probe_pt.sample(grid)

print("Point Probe at inner wall (x-axis):")
print(f"  Position: ({probe_location[0]}, {probe_location[1]}, {probe_location[2]})")
for name in sampled.point_data:
    val = sampled.point_data[name][0]
    if isinstance(val, np.ndarray) and val.size > 1:
        print(f"  {name}: {val}  (|v| = {np.linalg.norm(val):.6e})")
    else:
        print(f"  {name}: {float(val):.6e}")

In [ ]:
# ============================================================
# Line Probe: radial profile along x-axis (r = R_i to R_o)
# ============================================================
# Sample the stress field along the radial direction and compare
# with the Lamé analytical solution.

n_samples = 200
line = pv.Line(
    [inner_radius, 0, 0],     # start (inner wall)
    [outer_radius, 0, 0],     # end (outer wall)
    resolution=n_samples - 1,
)
sampled_line = line.sample(grid)

# Extract sampled positions and field values
r_sampled = np.array(sampled_line.points)[:, 0]  # x = r along this line
sig_xx_sampled = np.array(sampled_line.point_data["Stress_XX_nodal"])
ur_sampled = np.array(sampled_line.point_data["Radial_Disp"])

print(f"Line probe: {len(r_sampled)} samples from r={inner_radius} to r={outer_radius}")
print(f"  Stress_XX range: [{sig_xx_sampled.min():.2f}, {sig_xx_sampled.max():.2f}] MPa")

In [ ]:
# ============================================================
# Analytical solution (Lamé)
# ============================================================
# Along the x-axis (theta=0):
#   sigma_rr = sigma_xx = A + B/r^2
#   sigma_tt = sigma_yy = A - B/r^2
#   u_r = (1/E) * [(1+nu)*B/r + (1-2*nu)*(1+nu)*A*r]
#
# where A = p*a^2 / (b^2 - a^2),  B = -p*a^2*b^2 / (b^2 - a^2)

a, b = inner_radius, outer_radius
A_lame = p * a**2 / (b**2 - a**2)
B_lame = -p * a**2 * b**2 / (b**2 - a**2)

r_analytical = np.linspace(inner_radius, outer_radius, 500)
sig_rr_exact = A_lame + B_lame / r_analytical**2
sig_tt_exact = A_lame - B_lame / r_analytical**2
ur_exact = (1/E) * ((1+nu)*B_lame/r_analytical + (1-2*nu)*(1+nu)*A_lame*r_analytical)

print(f"Analytical at inner wall: \u03c3_rr = {sig_rr_exact[0]:.2f}, \u03c3_tt = {sig_tt_exact[0]:.2f} MPa")
print(f"Analytical at outer wall: \u03c3_rr = {sig_rr_exact[-1]:.2f}, \u03c3_tt = {sig_tt_exact[-1]:.2f} MPa")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Radial stress profile: FEM probe vs analytical
ax = axes[0]
ax.plot(r_analytical, sig_rr_exact, 'k-', lw=2, label=r'$\sigma_{rr}$ (Lamé)')
ax.plot(r_analytical, sig_tt_exact, 'k--', lw=2, label=r'$\sigma_{\theta\theta}$ (Lamé)')
ax.plot(r_sampled, sig_xx_sampled, 'ro', ms=3, alpha=0.5,
        label=r'$\sigma_{xx}$ (FEM probe)')
ax.set_xlabel('r [mm]')
ax.set_ylabel('Stress [MPa]')
ax.set_title('Radial Stress Profile — Line Probe vs Analytical')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Radial displacement: FEM probe vs analytical
ax = axes[1]
ax.plot(r_analytical, ur_exact, 'k-', lw=2, label=r'$u_r$ (Lamé)')
ax.plot(r_sampled, ur_sampled, 'bo', ms=3, alpha=0.5,
        label=r'$u_r$ (FEM probe)')
ax.set_xlabel('r [mm]')
ax.set_ylabel('Displacement [mm]')
ax.set_title('Radial Displacement — Line Probe vs Analytical')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(output_dir / "lame_probe_comparison.png"), dpi=150)
plt.show()
print(f"\nPlot saved: {output_dir / 'lame_probe_comparison.png'}")

## 5 — Plane Probe (Slice)

Cut the mesh with a plane to inspect field values on the cross-section.
Useful for 3D models where you want to see internal stress distributions.

For this 2D problem (z=0 plane), a radial cut line is more natural,
but we demonstrate the API for completeness.

In [ ]:
# Slice the mesh along the 45-degree line (normal = [-1, 1, 0])
# This cuts through the quarter annulus diagonally.

origin = np.array([150.0, 0.0, 0.0])  # midpoint of radial extent
normal = np.array([0.0, 1.0, 0.0])    # cut along y-direction

sliced = grid.slice(normal=normal, origin=origin)

if sliced.n_points > 0:
    print(f"Plane slice: {sliced.n_points} points, {sliced.n_cells} cells")
    print(f"Fields on slice: {list(sliced.point_data.keys())}")
    
    # Extract data along the slice for plotting
    slice_coords = np.array(sliced.points)
    slice_y = slice_coords[:, 1]
    slice_stress = np.array(sliced.point_data.get("Stress_XX_nodal", []))
    
    if len(slice_stress) > 0:
        # Sort by y for clean plotting
        order = np.argsort(slice_y)
        plt.figure(figsize=(8, 4))
        plt.plot(slice_y[order], slice_stress[order], 'g.-', lw=1.5)
        plt.xlabel('y [mm]')
        plt.ylabel(r'$\sigma_{xx}$ [MPa]')
        plt.title(f'Stress along plane slice at x = {origin[0]} mm')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("Slice returned no points (plane may not intersect the mesh).")

## 6 — Quick Inline Visualization with PyVista

If you don't need the full viewer GUI, PyVista can render directly
in the notebook (with `jupyter_backend='static'` or `'trame'`).

In [ ]:
# Static inline plot (works in any Jupyter environment)
pv.set_jupyter_backend('static')

plotter = pv.Plotter(off_screen=True, window_size=[900, 500])
plotter.set_background('#1a1a2e', top='#16213e')

plotter.add_mesh(
    grid,
    scalars='VonMises',
    cmap='turbo',
    show_edges=True,
    edge_color='#2C4A6E',
    line_width=1,
    scalar_bar_args={
        'title': 'Von Mises [MPa]',
        'color': 'white',
    },
)
plotter.add_axes(color='white')
plotter.view_xy()

# Save and display
img_path = str(output_dir / 'lame_vonmises.png')
plotter.screenshot(img_path)
plotter.close()

from IPython.display import Image, display
display(Image(filename=img_path, width=800))
print(f"Screenshot saved: {img_path}")

In [ ]:
# Deformed shape with displacement contour
plotter = pv.Plotter(off_screen=True, window_size=[900, 500])
plotter.set_background('#1a1a2e', top='#16213e')

# Undeformed reference (wireframe)
plotter.add_mesh(
    grid, style='wireframe', color='#888888',
    line_width=1, opacity=0.3, label='Undeformed',
)

# Deformed mesh (scale factor = 500 for visibility)
scale = 500.0
deformed = grid.copy()
deformed.points = np.array(grid.points) + scale * np.array(grid.point_data['Displacement'])

plotter.add_mesh(
    deformed,
    scalars='Radial_Disp',
    cmap='coolwarm',
    show_edges=True,
    edge_color='#803D00',
    line_width=1,
    scalar_bar_args={
        'title': f'u_r [mm] (x{int(scale)})',
        'color': 'white',
    },
)
plotter.add_axes(color='white')
plotter.view_xy()

img_path = str(output_dir / 'lame_deformed.png')
plotter.screenshot(img_path)
plotter.close()

display(Image(filename=img_path, width=800))
print(f"Deformed shape saved: {img_path}")

## Summary

The full pipeline from geometry to interactive visualization:

```
pyGmsh (geometry + mesh)
    │
    ├── renumber_mesh(method='rcm')   ← mutates Gmsh model
    │
    ├── get_fem_data()  →  FEMData    ← single data source
    │       │
    │       ├── fem.nodes() / fem.elements()  → OpenSees
    │       ├── fem.physical.get_nodes/get_elements()  → BCs & loads
    │       ├── fem.node_index() / fem.get_node_coords()  → lookups
    │       │
    │       └── OpenSees (solve)  →  numpy arrays
    │
    └── VTKExport(g)   ← while Gmsh is alive
            │
            ├── vtk.add_node_scalar/vector()
            ├── vtk.add_elem_scalar/vector()
            └── vtk.write("results.vtu")
                    │
                    ├── pyGmshViewer (Qt + VTK GUI with probes)
                    ├── PyVista (inline notebook plots)
                    └── ParaView (.vtu is native format)
```

**Key files generated:**
- `generated/lame_plate_results.vtu` — mesh + all result fields
- `generated/lame_probe_comparison.png` — FEM vs analytical comparison
- `generated/lame_vonmises.png` — Von Mises contour screenshot
- `generated/lame_deformed.png` — deformed shape screenshot

**v2 API changes vs v1:**
- No `get_numbered_mesh()` or `NumberedMesh` — replaced by `renumber_mesh()` + `get_fem_data()`
- No `tag_to_idx` / `gmsh_to_solver_node` maps — `fem.node_index()` and direct iteration
- No manual 0-based connectivity for VTK — `VTKExport(g)` handles it internally
- Boundary queries on `fem.physical` instead of `g.physical`